# Clase 4 — Introducción a **Polars** 
**Formato:** tutorial paso a paso — ejecutamos y discutimos cada celda.
**Objetivos de la clase**
- Conocer qué es Polars y por qué usarlo.
- Leer datos sencillos (CSV / Parquet).
- Inspeccionar un DataFrame: primeros registros, schema, tipos.
- Seleccionar columnas y filtrar filas.
- Crear/renombrar columnas y aplicar transformaciones.
- Detectar y manejar valores faltantes (fill / drop).
- Agrupar y agregar (groupby / agg).
- Exportar resultados y convertir entre Polars y Pandas.



## 1) Importo libs

In [1]:
import polars as pl
import pandas as pd
from pathlib import Path

## 2) Datos de entrada
- df_toy de juguete
- df_loca_arg con datos provenientes de un archivo csv
- df_loca_pan

In [2]:
df_toy = pl.DataFrame(
    {
        "departamento": ["San Jerónimo","La Capital","San Jerónimo","Garay", None],
        "localidad": ["Santa Fe","Rosario","Rafaela","Esperanza","SinNombre"],
        "poblacion": [100000, 1200000, 50000, None, 230],
        "lat": [-31.629, -32.946, -31.280, -31.448, None],
        "lon": [-60.707, -60.639, -61.619, -60.937, -60.5],
    }
)

In [3]:
df_toy

departamento,localidad,poblacion,lat,lon
str,str,i64,f64,f64
"""San Jerónimo""","""Santa Fe""",100000,-31.629,-60.707
"""La Capital""","""Rosario""",1200000,-32.946,-60.639
"""San Jerónimo""","""Rafaela""",50000,-31.28,-61.619
"""Garay""","""Esperanza""",null,-31.448,-60.937
null,"""SinNombre""",230,null,-60.5


- Dataset Argentina

In [30]:
path_loca_arg = Path.cwd().parent / "datos" / "localidades.csv"
df_loca_arg = pl.read_csv(path_loca_arg)

In [31]:
df_loca_arg.head()

categoria,centroide_lat,centroide_lon,departamento_id,departamento_nombre,fuente,id,localidad_censal_id,localidad_censal_nombre,municipio_id,municipio_nombre,nombre,provincia_id,provincia_nombre
str,f64,f64,i64,str,str,i64,i64,str,i64,str,str,i64,str
"""Entidad""",-31.396847,-64.059869,14014,"""Capital""","""INDEC""",1401401002,14014010,"""Córdoba""",140077,"""Córdoba""","""La Floresta""",14,"""Córdoba"""
"""Entidad""",-32.867313,-68.77954,50028,"""Guaymallén""","""INDEC""",5002802001,50028020,"""Guaymallén""",500028,"""Guaymallén""","""Bermejo""",50,"""Mendoza"""
"""Entidad""",-34.818755,-58.423388,6028,"""Almirante Brown""","""INDEC""",602801008,6028010,"""Almirante Brown""",60028,"""Almirante Brown""","""Malvinas Argentinas""",6,"""Buenos Aires"""
"""Entidad""",-34.844021,-58.362754,6028,"""Almirante Brown""","""INDEC""",602801009,6028010,"""Almirante Brown""",60028,"""Almirante Brown""","""Ministro Rivadavia""",6,"""Buenos Aires"""
"""Entidad""",-34.797373,-58.388453,6028,"""Almirante Brown""","""INDEC""",602801001,6028010,"""Almirante Brown""",60028,"""Almirante Brown""","""Adrogué""",6,"""Buenos Aires"""


- Dataset Panamá

In [34]:
path_loca_pan = Path.cwd().parent / "datos" / "panama" /"pa_localidades.json"
df_loca_pan = pl.read_json(path_loca_pan)

In [35]:
df_loca_pan

city,lat,lng,country,iso2,admin_name,capital,population,population_proper
str,str,str,str,str,str,str,str,str
"""Panama City""","""8.9833""","""-79.5167""","""Panama""","""PA""","""Panamá""","""primary""","""1500189""","""880691"""
"""San Miguelito""","""9.0330""","""-79.5000""","""Panama""","""PA""","""Panamá""","""minor""","""315019""","""315019"""
"""David""","""8.4333""","""-82.4333""","""Panama""","""PA""","""Chiriquí""","""admin""","""82907""","""82907"""
"""Colón""","""9.3572""","""-79.8986""","""Panama""","""PA""","""Colón""","""admin""","""78000""","""78000"""
"""Tocumen""","""9.0800""","""-79.3800""","""Panama""","""PA""","""Panamá""","""minor""","""74952""","""74952"""
…,…,…,…,…,…,…,…,…
"""La Ensenada""","""8.3660""","""-78.8464""","""Panama""","""PA""","""Panamá""","""minor""","""""",""""""
"""Jingurudó""","""7.8796""","""-78.0769""","""Panama""","""PA""","""Emberá-Wounaan""","""minor""","""""",""""""
"""Púcuro""","""7.9737""","""-77.4860""","""Panama""","""PA""","""Darién""","""minor""","""""",""""""


## 3) Exploración básica
- `head()` para ver primeras filas.
- `shape` para dimensiones (filas, columnas).
- `schema` para ver nombres y tipos.
- `describe()` para estadísticas rápidas (numéricas).

In [24]:
# Exploración rápida
print("Dimensiones (rows, cols):", df_toy.shape)
print("Schema:")
print(df_toy.schema)
print("Columnas:", df_toy.columns)

Dimensiones (rows, cols): (5, 5)
Schema:
Schema([('departamento', String), ('localidad', String), ('poblacion', Int64), ('lat', Float64), ('lon', Float64)])
Columnas: ['departamento', 'localidad', 'poblacion', 'lat', 'lon']


In [22]:
print("Descripción rápida (describe):")
print(df_toy.describe())

Descripción rápida (describe):
shape: (9, 6)
┌────────────┬──────────────┬───────────┬───────────────┬───────────┬──────────┐
│ statistic  ┆ departamento ┆ localidad ┆ poblacion     ┆ lat       ┆ lon      │
│ ---        ┆ ---          ┆ ---       ┆ ---           ┆ ---       ┆ ---      │
│ str        ┆ str          ┆ str       ┆ f64           ┆ f64       ┆ f64      │
╞════════════╪══════════════╪═══════════╪═══════════════╪═══════════╪══════════╡
│ count      ┆ 4            ┆ 5         ┆ 4.0           ┆ 4.0       ┆ 5.0      │
│ null_count ┆ 1            ┆ 0         ┆ 1.0           ┆ 1.0       ┆ 0.0      │
│ mean       ┆ null         ┆ null      ┆ 337557.5      ┆ -31.82575 ┆ -60.8804 │
│ std        ┆ null         ┆ null      ┆ 576402.576236 ┆ 0.760309  ┆ 0.442091 │
│ min        ┆ Garay        ┆ Esperanza ┆ 230.0         ┆ -32.946   ┆ -61.619  │
│ 25%        ┆ null         ┆ null      ┆ 50000.0       ┆ -31.629   ┆ -60.937  │
│ 50%        ┆ null         ┆ null      ┆ 100000.0      ┆ -31.44

## 4) Selección y filtrado
Operaciones comunes:
- Seleccionar columnas: `select()` o `df_toy[["col1","col2"]]`
- Filtrar por condiciones: `filter(pl.col("poblacion") > 1000)`
- Encadenar operaciones (lazy style o eager).

In [25]:
# Selección de columnas
subset = df_toy.select(["localidad","departamento","poblacion"])
display(subset.head())

# Filtrado simple: localidades con población mayor a 1000 (si existe la columna)

mask = pl.col("poblacion") > 1000
filtrado = df_toy.filter(mask)
print("\nFiltrado: poblacion > 1000 ->", filtrado.height, "filas")
display(filtrado.head(6))


localidad,departamento,poblacion
str,str,i64
"""Santa Fe""","""San Jerónimo""",100000
"""Rosario""","""La Capital""",1200000
"""Rafaela""","""San Jerónimo""",50000
"""Esperanza""","""Garay""",null
"""SinNombre""",null,230



Filtrado: poblacion > 1000 -> 3 filas


departamento,localidad,poblacion,lat,lon
str,str,i64,f64,f64
"""San Jerónimo""","""Santa Fe""",100000,-31.629,-60.707
"""La Capital""","""Rosario""",1200000,-32.946,-60.639
"""San Jerónimo""","""Rafaela""",50000,-31.28,-61.619


## 5) Crear y renombrar columnas
- `with_columns()` para agregar columnas.
- `alias()` para nombrar expresiones.
- `rename()` para renombrar columnas existentes.
Ejemplo: población en miles, y flag si la latitud está presente.


In [38]:
# Crear columnas nuevas
df2 = df_toy.with_columns(
    (pl.col("poblacion") / 1000).round(2).alias("pobl_miles"),
    pl.col("lat").is_null().alias("lat_missing")
)
display(df2.head())



departamento,localidad,poblacion,lat,lon,pobl_miles,lat_missing
str,str,i64,f64,f64,f64,bool
"""San Jerónimo""","""Santa Fe""",100000,-31.629,-60.707,100.0,false
"""La Capital""","""Rosario""",1200000,-32.946,-60.639,1200.0,false
"""San Jerónimo""","""Rafaela""",50000,-31.28,-61.619,50.0,false
"""Garay""","""Esperanza""",null,-31.448,-60.937,null,false
null,"""SinNombre""",230,null,-60.5,0.23,true


In [39]:
# Renombrar columnas (ejemplo)

df2 = df2.rename({"poblacion":"pobl"})
display(df2.head(3))

departamento,localidad,pobl,lat,lon,pobl_miles,lat_missing
str,str,i64,f64,f64,f64,bool
"""San Jerónimo""","""Santa Fe""",100000,-31.629,-60.707,100.0,false
"""La Capital""","""Rosario""",1200000,-32.946,-60.639,1200.0,false
"""San Jerónimo""","""Rafaela""",50000,-31.28,-61.619,50.0,false


## 6) Manejo de valores faltantes (nulls)
- Detectar nulos: `is_null()` y `sum()` por columna.
- Rellenar con un valor: `fill_null()` o `with_column(pl.col(...).fill_null(...))`
- Eliminar filas con nulos: `drop_nulls()` (en columnas o global).

In [42]:
# Rellenar nulos: ejemplo -> poblacion = 0, lat = promedio (si corresponde)
df_fill = df_toy
df_fill = df_fill.with_columns(pl.col("poblacion").fill_null(0))
mean_lat = float(df_fill.select(pl.col("lat").mean()).to_series()[0])
df_fill = df_fill.with_columns(pl.col("lat").fill_null(mean_lat))

print("\nDespués de fill_null:")
display(df_fill.head())


Después de fill_null:


departamento,localidad,poblacion,lat,lon
str,str,i64,f64,f64
"""San Jerónimo""","""Santa Fe""",100000,-31.629,-60.707
"""La Capital""","""Rosario""",1200000,-32.946,-60.639
"""San Jerónimo""","""Rafaela""",50000,-31.28,-61.619
"""Garay""","""Esperanza""",0,-31.448,-60.937
null,"""SinNombre""",230,-31.82575,-60.5


In [47]:
# Eliminar filas con nulos en columnas clave (ej: 'localidad' o 'departamento')
df_drop = df_toy.drop_nulls(subset=["departamento"])
print("\nFilas tras drop_nulls depto:", df_drop.height)


Filas tras drop_nulls depto: 4


In [48]:
df_drop

departamento,localidad,poblacion,lat,lon
str,str,i64,f64,f64
"""San Jerónimo""","""Santa Fe""",100000,-31.629,-60.707
"""La Capital""","""Rosario""",1200000,-32.946,-60.639
"""San Jerónimo""","""Rafaela""",50000,-31.28,-61.619
"""Garay""","""Esperanza""",null,-31.448,-60.937


## 7) Agrupación y agregación
- `group_by().agg()` para agrupar por departamento y obtener conteos, promedio de población, etc.
- Ejemplo práctico: contar localidades por departamento y obtener población total/promedio.

In [13]:
# Agrupar por departamento y agregar

ag = df_toy.group_by("departamento").agg([
    pl.len().alias("n_localidades"),
    pl.col("poblacion").mean().alias("pobl_media"),
    pl.col("poblacion").sum().alias("pobl_total"),
])

# Mostrar ordenado por número de localidades (desc)
ag_sorted = ag.sort("n_localidades", descending=True)
display(ag_sorted.head(20))

departamento,n_localidades,pobl_media,pobl_total
str,u32,f64,i64
"""San Jerónimo""",2,75000.0,150000
null,1,230.0,230
"""Garay""",1,null,0
"""La Capital""",1,1.2e6,1200000


## 8) Ordenamiento y top-N
- `sort()` para ordenar por columnas.
- `head(n)` para obtener top-n. 

In [16]:
# Top-N: localidades con mayor población (si existe)

top = df_toy.sort("poblacion", descending=True).select(["localidad","departamento","poblacion"]).head(10)
display(top)


localidad,departamento,poblacion
str,str,i64
"""Esperanza""","""Garay""",null
"""Rosario""","""La Capital""",1200000
"""Santa Fe""","""San Jerónimo""",100000
"""Rafaela""","""San Jerónimo""",50000
"""SinNombre""",null,230


## 9) Joins
- `join()` para combinar tablas (left, inner, outer).
- Ejemplo: unir con una tabla pequeña de referencia (creada aquí) que mapea departamento -> región.


In [19]:
# Ejemplo de join con tabla de referencia
ref = pl.DataFrame({
    "departamento": df_toy.select("departamento").unique().to_series().to_list(),
    "region": ["Región "+str(i%3+1) for i in range(df_toy.select("departamento").unique().height)]
})



In [20]:
ref

departamento,region
str,str
"""Garay""","""Región 1"""
null,"""Región 2"""
"""La Capital""","""Región 3"""
"""San Jerónimo""","""Región 1"""


In [21]:
joined = df_toy.join(ref, on="departamento", how="left")
print("Join realizado. Columnas resultantes:")
display(joined.head())

Join realizado. Columnas resultantes:


departamento,localidad,poblacion,lat,lon,region
str,str,i64,f64,f64,str
"""San Jerónimo""","""Santa Fe""",100000,-31.629,-60.707,"""Región 1"""
"""La Capital""","""Rosario""",1200000,-32.946,-60.639,"""Región 3"""
"""San Jerónimo""","""Rafaela""",50000,-31.28,-61.619,"""Región 1"""
"""Garay""","""Esperanza""",null,-31.448,-60.937,"""Región 1"""
null,"""SinNombre""",230,null,-60.5,null


## 10) Convertir a Pandas y exportar resultados
- `to_pandas()` para pasar a pandas cuando necesites compatibilidad.
- `write_csv()` / `write_parquet()` para guardar resultados.


In [49]:
# Convertir a pandas (ejemplo)

pdf = df_toy.to_pandas()


df_toy.write_csv(out_path)
print("Resultado guardado en:", out_path)


NameError: name 'out_path' is not defined

## 11) Ejercicios
1. Filtrar todas las localidades del departamento 'La Capital' (o el que corresponda en tu dataset).  
2. Contar cuántas localidades hay por departamento y mostrar las 5 con más localidades.  
3. Rellenar los valores faltantes de `poblacion` con 0 y crear una columna `pobl_miles` (población en miles, redondeada a 2 decimales).  
4. Guardar el resultado limpio en CSV.


## 12) Recursos adicionales
- Documentación oficial Polars: https://pola-rs.org/ (buscar 'polars' en Python).  
- Conversión entre Polars y Pandas: `to_pandas()` / `pl.from_pandas()`.
- Si usan datasets grandes: explorar el modo *lazy* de Polars (`pl.scan_csv()` / `df_toy.lazy()`).